# Parts 3–5 Demo: Parameter Refinement, Vetting, and Baseline Classification

This notebook continues the project after preprocessing and detection. It shows how a candidate periodic dip becomes a refined, vetted, classified signal.

## Design idea

- Part 3 refines period, epoch, duration, depth, SNR, and rough uncertainties.
- Part 4 extracts vetting features: odd/even, secondary eclipse, centroid shift, CROWDSAP, shape, and noise.
- Part 5 applies a transparent rule-based baseline classifier.

The supervised AI classifier should later be trained on the curated labeled dataset using this feature table.

In [ ]:
from exoplanet_pipeline import PipelineConfig
from exoplanet_pipeline.synthetic import make_synthetic_transit_lc, make_synthetic_eb_lc, make_synthetic_blend_lc
from exoplanet_pipeline.pipeline import run_parts_1_to_5_from_raw

cfg = PipelineConfig(detection_method="bls", n_periods=900, n_durations=8, detection_use_variants=False)

samples = {
    "planet": make_synthetic_transit_lc(noise_ppm=300, random_seed=42),
    "eclipsing_binary": make_synthetic_eb_lc(noise_ppm=500, random_seed=43),
    "blend": make_synthetic_blend_lc(noise_ppm=350, random_seed=44),
}

results = {name: run_parts_1_to_5_from_raw(raw, cfg) for name, raw in samples.items()}
for name, res in results.items():
    print(name, res["clean"].status, res["detection"].status, res["catalog"].shape)

In [ ]:
import pandas as pd

catalog = pd.concat([res["catalog"].assign(sample=name) for name, res in results.items() if not res["catalog"].empty], ignore_index=True)
cols = [
    "sample", "period_days", "depth_ppm", "local_snr",
    "fit_period_days", "fit_duration_days", "fit_depth_ppm", "fit_snr",
    "vet_secondary_sigma", "vet_odd_even_sigma", "vet_centroid_shift_sigma", "vet_crowding_risk",
    "class_predicted_class", "class_confidence", "class_evidence"
]
catalog[cols]

## Diagnostic plot for one sample

In [ ]:
from exoplanet_pipeline.diagnostics import plot_vetting_summary

name = "planet"
res = results[name]
plot_vetting_summary(
    res["clean"],
    res["detection"].candidates[0],
    res["fit_results"][0],
    res["vetting_results"][0],
    res["classification_results"][0],
);

## What to do next

The next stage is Part 6: train a supervised AI classifier on the curated labeled dataset. Use the candidate/fit/vetting catalog as the tabular feature table. The rule-based classifier remains the baseline for comparison.